# Time Series Course — m11-fl2 (block 3)

_From the **Time Series & Forecasting** course on Zylo._

Run all cells (`Runtime → Run all` or `Ctrl+F9`) to execute.

In [ ]:
# Transformer model for 7-day ahead forecasting
import torch
import torch.nn as nn
import math

class EnergyTransformer(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=3,
                 lookback=168, horizon=168):
        super().__init__()
        self.proj = nn.Linear(3, d_model)  # demand + hour_sin + hour_cos
        pe = torch.zeros(lookback, d_model)
        pos = torch.arange(lookback).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model*4, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, horizon)

    def forward(self, x):
        x = self.proj(x) + self.pe
        x = self.encoder(x)
        return self.head(x.mean(dim=1))

# Training: create windows with (demand, hour_sin, hour_cos) features
# Evaluate on last 14 days: first 7 days = input, last 7 days = target